# Multi-objective optimization of Community-1

Speaker diarization purity and coverage usually conflict: splitting speakers into more clusters tends to improve purity, while merging clusters tends to improve coverage. This tutorial tunes `pyannote/speaker-diarization-community-1` for both metrics and retrieves the Pareto front—the operating points for which neither metric can improve without degrading the other.

You need a Hugging Face token with access to Community-1 and a [`pyannote.database`](https://github.com/pyannote/pyannote-database) protocol with reference annotations.

In [ ]:
from pathlib import Path
from types import MethodType

import matplotlib.pyplot as plt
import torch
from pyannote.audio import Pipeline
from pyannote.database import FileFinder, registry
from pyannote.metrics.diarization import DiarizationCoverage, DiarizationPurity
from pyannote.pipeline import Optimizer

## Optimize from the command line

Once Community-1's `config.yaml` is available locally, repeat `--metric` to define a multi-objective study. The CLI keeps updating a `*.pareto.yaml` manifest containing every non-dominated trial, its purity and coverage, and the corresponding pipeline parameters. A single `--metric` keeps the existing best-pipeline YAML behavior.

In [ ]:
# Run this in a shell, replacing the registry and protocol.
# pyannote-audio optimize community1/config.yaml \
#   MyDatabase.SpeakerDiarization.MyProtocol \
#   --registry /path/to/database.yml \
#   --metric DiarizationPurity \
#   --metric DiarizationCoverage \
#   --average-case --max-iterations 100

## Load the development set

Replace the database configuration and protocol names below. Optimization must use a development set rather than the final test set.

In [ ]:
registry.load_database("/path/to/database.yml")
protocol = registry.get_protocol(
    "MyDatabase.SpeakerDiarization.MyProtocol",
    preprocessors={"audio": FileFinder()},
)
development_files = list(protocol.development())

## Turn Community-1 into a two-objective pipeline

`Optimizer` accepts a sequence of metrics when `get_direction` returns one direction per metric. Community-1 normally returns a rich output object; `legacy = True` makes it return the `Annotation` expected by `pyannote.metrics`.

In [ ]:
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token="HUGGINGFACE_ACCESS_TOKEN",
)
pipeline.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
pipeline.legacy = True
pretrained_params = pipeline.parameters(instantiated=True)


def get_metric(self):
    return DiarizationPurity(), DiarizationCoverage()


def get_direction(self):
    return "maximize", "maximize"


pipeline.get_metric = MethodType(get_metric, pipeline)
pipeline.get_direction = MethodType(get_direction, pipeline)

## Optimize and obtain the Pareto front

The journal lets an interrupted run resume and allows multiple optimization processes to share the study. `average_case=True` uses the purity and coverage accumulated over the whole development set. Omit it to optimize the lower 90% confidence bound of each metric.

In [ ]:
optimizer = Optimizer(
    pipeline,
    db=Path("community1-purity-coverage.journal"),
    study_name="community1-purity-coverage",
    average_case=True,
)
optimizer.tune(
    development_files,
    n_iterations=100,
    warm_start=pretrained_params,
)

pareto_front = sorted(
    optimizer.pareto_front, key=lambda point: point["values"][0]
)
pareto_front

Every entry contains the Optuna trial number, `(purity, coverage)`, and the corresponding nested Community-1 parameters. The metric order matches `get_metric`.

In [ ]:
purity = [point["values"][0] for point in pareto_front]
coverage = [point["values"][1] for point in pareto_front]

fig, ax = plt.subplots()
ax.plot(purity, coverage, "o-")
ax.set(xlabel="Diarization purity", ylabel="Diarization coverage")
ax.grid(True)

## Select an operating point

There is deliberately no single `best_params` in a multi-objective study. Choose a point according to the application. The example below selects the point with the best worst metric (a max-min rule), then restores Community-1's rich output before using it.

In [ ]:
selected = max(pareto_front, key=lambda point: min(point["values"]))
optimized_pipeline = pipeline.instantiate(selected["params"])
optimized_pipeline.legacy = False

print(f"purity={selected['values'][0]:.3f}")
print(f"coverage={selected['values'][1]:.3f}")
print(selected["params"])